# LexChain LLM legal-analysis benchmark — 3 models, one API path

**Research question:** which model is best at legal analysis. **Single variable = the model.** All three are ~70B-tier instruct models from different families, accessed through the **same OpenRouter API** under a **zero-data-retention** policy. API-only deployment, no self-hosting — this removes the size-bias confound.

| Model | Family | Tier note |
|---|---|---|
| `meta-llama/llama-3.3-70b-instruct` | Meta | 70B dense |
| `qwen/qwen-2.5-72b-instruct` | Qwen | 72B dense (32k context — tightest, still clears our 21.6k worst case) |
| `mistralai/mixtral-8x22b-instruct` | Mistral | MoE 141B total / ~39B active; ~70B-class compute, older (Apr 2024) |

**Privacy:** every request sends `provider = {data_collection:"deny", allow_fallbacks:false, zdr:true}` — zero data retention, and it **fails loudly rather than falling back to a non-compliant provider**. Also turn **prompt logging OFF** in your OpenRouter account (Settings → Privacy); the request flags cover provider-side retention, the account setting covers OpenRouter's own logging.

Everything else is identical to the frozen harness: prompt v2.0-cuad-checklist, temperature 0, same JSON schema/parser, same 10 seed-42 docs, same `ground_truth_key.csv` (reuse it unchanged), same blind build (Output A/B/C, SummEval dims) + entity/risk F1 + aggregation.

**No GPU needed** — a CPU runtime is fine. Set Colab secret `OPENROUTER_API_KEY` before starting.

In [ ]:
# Cell 1 — mount Drive, clone repo, install deps, load OpenRouter key
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
if not os.path.exists('/content/lexchain-chunk-embed-bench'):
    !git clone https://github.com/MarvinPescos/lexchain-chunk-embed-bench.git /content/lexchain-chunk-embed-bench
%cd /content/lexchain-chunk-embed-bench
!git pull
!pip install -q -r requirements-analysis.txt
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
print('OpenRouter key loaded:', bool(os.environ.get('OPENROUTER_API_KEY')))

In [ ]:
# Cell 2 — remove stale checkpoints from the previous (self-hosted) model set,
# so no de-registered model can leak into the blind sheet or results
import glob, json, os
from analysis.data import MODELS
base = '/content/drive/MyDrive/lexchain_bench/analysis'
removed = 0
for f in glob.glob(f'{base}/analyses/*.json'):
    if json.load(open(f))['model'] not in MODELS:
        os.remove(f); removed += 1
for f in glob.glob(f'{base}/raw_failures/*'):
    os.remove(f)
print(f'removed {removed} stale checkpoints; current models: {list(MODELS)}')

In [ ]:
# Cell 3 — data + ground-truth authoring materials + CONTEXT SAFETY CHECK
# (no model output is produced or shown here)
!python download_data.py
!python -m analysis.prepare_ground_truth

import sys; sys.path.insert(0, '.')
from analysis.data import load_docs, sample_stems, context_check
docs = load_docs(); stems = sample_stems(docs.keys(), n=10)
report = context_check({s: docs[s] for s in stems})  # raises loudly on any misfit
print('CONTEXT CHECK PASSED — all 10 docs fit all models.')
for m, info in report['per_model'].items():
    print(f"  {m}: worst case {info['worst_case_tokens']:,} / limit {info['limit']:,}")

## ✋ GT-AUTHORING GATE — reuse your existing key

If you already authored `ground_truth_key.csv`, it is **unchanged and reusable** (same 10 docs). Confirm it is in `MyDrive/lexchain_bench/analysis/`. If not, author it now from `doc_texts/*.txt` (blind to model outputs) and save as `ground_truth_key.csv`.

Cells 4–5 only produce checkpoints (status/latency logs, no content). Cell 6 — the first cell that *renders* outputs — refuses to run until the key exists.

In [ ]:
# Cell 4 — smoke: 2 docs x 3 models (6 calls); logs show status/latency only
!python -m analysis.analyze --smoke 2

import json
from pathlib import Path
cache = Path('/content/drive/MyDrive/lexchain_bench/analysis')
recs = [json.loads(p.read_text()) for p in (cache/'analyses').glob('*.json')]
recs = [r for r in recs if r['model'] in __import__('analysis.data', fromlist=['MODELS']).MODELS]
ok = sum(r.get('ok', False) for r in recs)
print(f"{ok}/{len(recs)} outputs parsed + schema-valid")
by_model, proj = {}, 0
for r in recs:
    by_model.setdefault(r['model'], []).append(r.get('latency_s') or 0)
for m, ls in sorted(by_model.items()):
    mean = sum(ls)/len(ls); proj += mean*10
    print(f"  {m}: {mean:.1f}s/doc ({r['latency_label']}) -> ~{mean*10/60:.1f} min for 10 docs")
print(f"Projected full run (30 calls): ~{proj/60:.0f} min")

In [ ]:
# Cell 5 — full run: 10 docs x 3 models = 30 outputs (resumable; reruns skip)
!python -m analysis.analyze

In [ ]:
# Cell 6 — blind sheet (first output-rendering step; guardrail-enforced)
!python -m analysis.build_blind_eval
print('\nblind_eval.csv: 30 rows, Output A–C labels, SummEval columns')
print('Rate coherence / consistency / fluency / relevance (1–5) + hallucination notes.')
print('Do NOT open unblinding_key.csv until all rating is done.')

### Human step (off-Colab)
Rate all 30 rows of `blind_eval.csv` on the **SummEval** dimensions: coherence, consistency, fluency, relevance (1–5). Then run Cell 7.

In [ ]:
# Cell 7 — aggregate: un-blind, entity/risk F1 vs your key, wins, final table
!python -m analysis.aggregate_analysis